# Atomic hydrogen

This tutorial solves the `hydrogen/atomic/basic` model: the populations of H(n) in a plasma of given electron temperature and density, the processes that set them, and the effective (collisional-radiative) ionization and recombination coefficients. The physics behind the solve is in [The CR model](/OpenCR/guides/model/).

In [ ]:
import numpy as np
import plasma_plots
import xarray as xr

import opencr

## The model

`load_model` builds the network; its manifest lists the states, reactions, sources and assumptions.

In [ ]:
model = opencr.load_model("hydrogen/atomic/basic", n_max=20)
manifest = model.manifest()
print(model)
print(manifest["counts"]["by_kind"])
for assumption in manifest["assumptions"][:3]:
    print("-", assumption)

## A quasi-static solve

The ground state and the proton are the *drivers*: their densities are inputs. The excited states are in steady state for them.

In [ ]:
conditions = opencr.Conditions(
    electron_temperature=3.0,  # eV
    electron_density=1e19,  # m^-3
    densities={"H(n=1)": 1e19, "H+": 1e18},
)
result = model.solve_steady_state(conditions)
print(result.report.summary())

The populations divided by their statistical weights, against the level energy. A Boltzmann distribution at $T_e$ would be a straight line of slope $-1/T_e$; the low levels fall below it (radiative losses), the high ones approach Saha equilibrium with the ions.

In [ ]:
energies = [s.energy for s in model.network.states[:-1]]
reduced = xr.DataArray(
    [result.populations[s.name] / s.weight for s in model.network.states[:-1]],
    coords={
        "energy": ("energy", energies, {"units": "eV", "long_name": "level energy"})
    },
    dims="energy",
    name="n/g",
    attrs={"units": "m^-3"},
)
with plasma_plots.figure(1, 1) as fig:
    reduced.plasma.plot.lineout(ax=fig[0], title="H(n) populations / g_n")
    fig[0].set_yscale("log")

## Which processes matter

The largest gains and losses of H(n=3):

In [ ]:
for entry in result.dominant_processes("H(n=3)", 6):
    print(
        f"{entry['equation']:28s} {entry['rate']:+.3e} m^-3/s  ({entry['fraction']:.0%})"
    )

## Effective coefficients

Eliminating the excited states leaves effective rates between the drivers: $S_\mathrm{CR}$ from H(n=1) to H⁺ and $\alpha_\mathrm{CR}$ back. `scan` computes them on a grid. At high density, stepwise ionization through excited states raises $S_\mathrm{CR}$ above the ground-state rate, and three-body recombination raises $\alpha_\mathrm{CR}$.

In [ ]:
temperatures = np.geomspace(1.0, 100.0, 25)
data = opencr.scan(
    model,
    conditions,
    lines=False,
    electron_temperature=temperatures,
    electron_density=[1e17, 1e19, 1e21],
)
s_cr = data["effective_rate"].sel(target="H+", source="H(n=1)")
alpha_cr = data["effective_rate"].sel(target="H(n=1)", source="H+")
ground = xr.DataArray(
    [
        model.rate_coefficients(conditions.replace(electron_temperature=t))["ion:1"]
        for t in temperatures
    ],
    coords={"electron_temperature": temperatures},
    dims="electron_temperature",
)

In [ ]:
with plasma_plots.figure(1, 2) as fig:
    for ne in data["electron_density"].values:
        s_cr.sel(electron_density=ne).plasma.plot.lineout(ax=fig[0], title="ionization")
        fig[0].lines[-1].set_label(f"S_CR, ne={ne:.0e}")
        alpha_cr.sel(electron_density=ne).plasma.plot.lineout(
            ax=fig[1], title="recombination"
        )
        fig[1].lines[-1].set_label(f"alpha_CR, ne={ne:.0e}")
    fig[0].plot(temperatures, ground, "k--", label="ground state only")
    for ax in fig:
        ax.set_xscale("log")
        ax.set_yscale("log")
        ax.set_ylabel("m^3/s")
        ax.legend()